In [1]:
!pip -q install openai
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('GEMINI_API_KEY'),
                base_url="https://generativelanguage.googleapis.com/v1beta/openai/")
names = [m.id for m in client.models.list()]
print([n for n in names if 'flash' in n.lower()][:15])

['models/gemini-2.5-flash', 'models/gemini-2.5-flash-preview-tts', 'models/gemini-flash-latest', 'models/gemini-flash-lite-latest', 'models/gemini-2.5-flash-lite', 'models/gemini-2.5-flash-image', 'models/gemini-3-flash-preview', 'models/gemini-3.1-flash-lite-preview', 'models/gemini-3.1-flash-lite', 'models/gemini-3.1-flash-image-preview', 'models/gemini-3.1-flash-image', 'models/gemini-3.1-flash-lite-image', 'models/gemini-3.5-flash', 'models/gemini-3.5-flash-lite', 'models/gemini-omni-flash-preview']


In [4]:
MODEL = "gemini-3.5-flash-lite"
r = client.chat.completions.create(
    model=MODEL, max_tokens=1000,
    messages=[{"role": "user", "content": "Jelaskan dalam dua kalimat apa itu dose-volume histogram."}])
print(r.choices[0].finish_reason, r.usage)
print(r.choices[0].message.content)

stop CompletionUsage(completion_tokens=75, prompt_tokens=14, total_tokens=89, completion_tokens_details=None, prompt_tokens_details=None)
*Dose-volume histogram* (DVH) adalah grafik yang menunjukkan hubungan antara dosis radiasi yang diberikan dan volume jaringan tubuh (seperti organ sehat atau tumor) yang menerima dosis tersebut. Grafik ini digunakan oleh dokter untuk mengevaluasi dan merencanakan terapi radiasi agar tumor hancur secara optimal sambil meminimalkan kerusakan pada jaringan sehat di sekitarnya.


# unduh korpus dokumen

In [5]:
import csv, time, requests
from pathlib import Path
from google.colab import drive
drive.mount('/content/drive')
out = Path('/content/drive/MyDrive/medphys-rag'); (out / 'xml').mkdir(parents=True, exist_ok=True)

Mounted at /content/drive


In [ ]:
BASE = "https://www.ebi.ac.uk/europepmc/webservices/rest"
OK = {'cc by', 'cc0'}
PER_TOPIC = 3
TOPICS = [
    "IMRT quality assurance gamma analysis",
    "brachytherapy cervical cancer dosimetry",
    "adaptive radiotherapy",
    "deep learning auto-segmentation radiotherapy organs at risk",
    "CT dose optimization diagnostic reference levels",
    "Monte Carlo dose calculation radiotherapy",
    "radiation protection occupational exposure",
    "stereotactic body radiotherapy motion management",
    "proton therapy range uncertainty",
    "cone beam CT image-guided radiotherapy",
    "MRI-guided radiotherapy",
    "machine learning radiotherapy treatment planning",
]

rows, seen = [], set()
for t in TOPICS:
    q = f'({t}) AND OPEN_ACCESS:y AND HAS_FT:Y AND PUB_YEAR:[2018 TO 2026] AND LANG:eng'
    try:
        res = requests.get(f"{BASE}/search", params=dict(query=q, resultType='core', format='json', pageSize=25), timeout=60).json()
    except Exception as e:
        print('gagal cari:', t, e); continue
    n = 0
    for a in res.get('resultList', {}).get('result', []):
        lic, pmcid = (a.get('license') or '').lower().strip(), a.get('pmcid')
        if lic not in OK or not pmcid or pmcid in seen: continue
        try:
            x = requests.get(f"{BASE}/{pmcid}/fullTextXML", timeout=60)
        except Exception as e:
            continue
        if x.status_code != 200 or len(x.text) < 30000: continue   # lewati teks pendek (editorial, dll.)
        (out / 'xml' / f'{pmcid}.xml').write_text(x.text, encoding='utf-8')
        rows.append(dict(pmcid=pmcid, doi=a.get('doi'), title=a.get('title'), year=a.get('pubYear'),
                         journal=((a.get('journalInfo') or {}).get('journal') or {}).get('title'),
                         license=a.get('license'), topic=t, url=f"https://europepmc.org/article/PMC/{pmcid}"))
        seen.add(pmcid); n += 1; time.sleep(0.2)
        if n >= PER_TOPIC: break
    print(f'{n} dokumen | {t}')

with open(out / 'sources.csv', 'w', newline='', encoding='utf-8') as f:
    w = csv.DictWriter(f, fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)
print('total:', len(rows), '| lisensi:', {r['license'] for r in rows})

# baca XML, potong menjadi chunk, simpan

In [6]:
import json, re, pandas as pd
from xml.etree import ElementTree as ET

src = pd.read_csv(out / 'sources.csv')
print(src[['pmcid', 'year', 'title']].to_string(max_colwidth=90))

def text_of(el): return ' '.join(''.join(el.itertext()).split())

def walk(node, path, acc):
    for ch in node:
        tag = ch.tag.split('}')[-1]
        if tag == 'sec':
            t = ch.find('title')
            title = text_of(t) if t is not None else ''
            walk(ch, path + ([title] if title else []), acc)
        elif tag == 'p':
            txt = text_of(ch)
            if txt: acc.append((' > '.join(path) or 'Body', txt))

def split_long(txt, limit):
    parts, cur = [], ''
    for s in re.split(r'(?<=[.!?])\s+', txt):
        if cur and len(cur) + len(s) > limit: parts.append(cur); cur = s
        else: cur = (cur + ' ' + s).strip()
    return parts + ([cur] if cur else [])

MAX_CH, MIN_CH = 1500, 200
chunks = []
for _, r in src.iterrows():
    root = ET.parse(out / 'xml' / f"{r['pmcid']}.xml").getroot()
    acc = []
    ab = root.find('.//abstract')
    if ab is not None: walk(ab, ['Abstract'], acc)
    body = root.find('.//body')
    if body is not None: walk(body, [], acc)

    packed, cur_sec, buf = [], None, ''
    for sec, txt in acc:
        for piece in (split_long(txt, MAX_CH) if len(txt) > MAX_CH else [txt]):
            if sec == cur_sec and buf and len(buf) + len(piece) + 1 <= MAX_CH:
                buf += ' ' + piece
            else:
                if buf: packed.append((cur_sec, buf))
                cur_sec, buf = sec, piece
    if buf: packed.append((cur_sec, buf))
    for i, (sec, txt) in enumerate(packed):
        if len(txt) >= MIN_CH:
            chunks.append(dict(chunk_id=f"{r['pmcid']}_{i}", pmcid=r['pmcid'],
                               title=r['title'], section=sec, text=txt))

with open(out / 'chunks.jsonl', 'w', encoding='utf-8') as f:
    for c in chunks: f.write(json.dumps(c, ensure_ascii=False) + '\n')

lens = [len(c['text']) for c in chunks]
print('\nchunk:', len(chunks), '| rata-rata panjang:', int(sum(lens) / len(lens)),
      '| dokumen:', len({c['pmcid'] for c in chunks}))
print(pd.Series([c['section'].split(' > ')[0] for c in chunks]).value_counts().head(12))
print('\nContoh:', chunks[len(chunks) // 2]['section'], '\n', chunks[len(chunks) // 2]['text'][:500])

          pmcid  year                                                                                      title
0   PMC13714354  2026  Daily Multi-Leaf Collimator positional error prediction using machine learning: A Dyna...
1   PMC13175351  2026  Analysis of IMRT patient specific quality assurance using EPID measurement data from t...
2   PMC13601935  2026  Assessing the role of plan complexity and target geometry through multi-institutional ...
3   PMC13708988  2026              Polymer-Based Additive Manufacturing in Cervical Cancer Management: A Review.
4   PMC12885748  2026  Impact of rectal gas evacuation on dosimetry and applicator displacement in cervical c...
5   PMC13437463  2026  Prognostic factors and outcomes of CT-guided Iodine-125 seed brachytherapy for retrope...
6   PMC13703279  2026  CBCT-Guided Online Adaptive Ultrahypofractionated Whole-Breast Radiotherapy for Early-...
7   PMC13565088  2026   MR-Guided Adaptive Radiotherapy in Lung Cancer: Current Landscape and Fu

# embedding, indeks FAISS, BM25, dan uji cepat

In [7]:
!pip -q install sentence-transformers faiss-cpu rank-bm25
import json, re, numpy as np, faiss
from sentence_transformers import SentenceTransformer
from rank_bm25 import BM25Okapi

DROP = {'PMC13402420'}   # editorial, bukan konten teknis
chunks = [json.loads(l) for l in open(out / 'chunks.jsonl', encoding='utf-8')]
chunks = [c for c in chunks if c['pmcid'] not in DROP]
print(len(chunks), 'chunk,', len({c['pmcid'] for c in chunks}), 'dokumen')

emb_model = SentenceTransformer('intfloat/multilingual-e5-small')
texts = ['passage: ' + c['title'] + '. ' + c['section'] + '. ' + c['text'] for c in chunks]
E = emb_model.encode(texts, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
index = faiss.IndexFlatIP(E.shape[1]); index.add(E.astype('float32'))
np.save(out / 'embeddings_e5small.npy', E)

tok = lambda s: re.findall(r'\w+', s.lower())
bm25 = BM25Okapi([tok(c['title'] + ' ' + c['section'] + ' ' + c['text']) for c in chunks])

def dense(q, k=5):
    qv = emb_model.encode(['query: ' + q], normalize_embeddings=True).astype('float32')
    s, i = index.search(qv, k)
    return [(int(j), float(x)) for j, x in zip(i[0], s[0])]

def sparse(q, k=5):
    sc = bm25.get_scores(tok(q)); top = np.argsort(-sc)[:k]
    return [(int(j), float(sc[j])) for j in top]

def show(res):
    for j, s in res:
        print(f"{s:.3f} | {chunks[j]['pmcid']} | {chunks[j]['section'][:45]} | {chunks[j]['text'][:100]}")

for q in ["What causes MLC positional errors and how can they be predicted?",
          "Apa keuntungan adaptive radiotherapy pada kanker prostat?",
          "Who won the 2022 World Cup?"]:
    print('\n', q); print('DENSE'); show(dense(q)); print('BM25'); show(sparse(q))

1277 chunk, 33 dokumen


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/498k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/655 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  471MB            

... (7 outputs left out)

special_tokens_map.json:   0%|          | 0.00/167 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

Batches:   0%|          | 0/20 [00:00<?, ?it/s]


 What causes MLC positional errors and how can they be predicted?
DENSE
0.905 | PMC13714354 | DISCUSSION | In this study, we proposed using the LightGBM machine learning algorithm to predict daily MLC errors
0.901 | PMC13714354 | Abstract > Purpose | This study proposes a machine learning (ML) approach to predict daily MLC positional deviations usin
0.898 | PMC13714354 | Abstract > Background | Accurate positioning of multi‐leaf collimators (MLCs) is critical for ensuring the dosimetric precis
0.894 | PMC13714354 | INTRODUCTION | Despite these advances, an important gap remains in the application of machine learning to daily MLC
0.891 | PMC13714354 | Abstract > Methods | DICOM‐RT and Dynalog files from 29 IMRT patients were collected, and 78 features were extracted, inc
BM25
24.449 | PMC13714354 | DISCUSSION | 46 Despite the advantages of the proposed method, as they predicted average leaf positional errors, 
23.472 | PMC13714354 | INTRODUCTION | It is important to note that changes i

# pastikan 10 soal “tak terjawab” memang tak terjawab

In [8]:
import re
terms = {'Q041': r'lens of the eye|eye lens|lens dose|dose limit', 'Q042': r'panoramic',
         'Q043': r'lead equivalen|lead apron', 'Q044': r'specific absorption rate|\bSAR\b',
         'Q045': r'thermal index|mechanical index', 'Q046': r'occupancy factor|use factor',
         'Q047': r'\bSUV', 'Q048': r'hippocamp', 'Q049': r'TG-?43', 'Q050': r'glandular'}
for q, pat in terms.items():
    hits = [c for c in chunks if re.search(pat, c['text'], re.I)]
    print(q, len(hits), 'chunk |', sorted({c['pmcid'] for c in hits})[:6])
    for c in hits[:2]:
        m = re.search(pat, c['text'], re.I)
        print('    ...', c['text'][max(0, m.start() - 80): m.end() + 120].replace('\n', ' '))

Q041 8 chunk | ['PMC13163279', 'PMC13437463', 'PMC13565088', 'PMC13664887', 'PMC13669425']
    ... ons. The OAR are mainly medulla spinalis, intestine, ureter, bladder, etc., The dose limit of endangering organs is referenced from the dose parameters of endangering organs in the radioactive particle therapy 
    ... EQD2-based organ-at-risk constraints to minimize severe side effects. Suggested dose limits include esophageal V60 < 40% and Dmax < 100 Gy, lung V20 < 40%, proximal bronchial tree/trachea Dmax < 110 Gy, aorta a
Q042 0 chunk | []
Q043 9 chunk | ['PMC13617968']
    ... , 26% preferred not to say). Only 50% reported wearing an appropriately fitting lead apron, while 41% reported inadequate fit. Consistent use of a thyroid shield was reported by 41% of respondents, whereas 49% 
    ... , including sex and university. (2) PPE, including perceived appropriateness of lead apron fit and use of thyroid protection. (3) Radiation safety behaviour and guidance, including whether studen

In [9]:
checks = {'Q041': r'lens of the eye|eye lens|lens of eye|\blens\b.{0,40}(dose|limit)',
          'Q043': r'lead equivalen|mm\s*Pb|mm of lead|0\.25 ?mm|0\.35 ?mm',
          'Q048': r'hippocamp'}
for q, pat in checks.items():
    hits = [c for c in chunks if re.search(pat, c['text'], re.I)]
    print(q, len(hits), 'chunk |', sorted({c['pmcid'] for c in hits}))
    for c in hits[:3]:
        m = re.search(pat, c['text'], re.I)
        print('   ...', c['text'][max(0, m.start() - 150): m.end() + 250].replace('\n', ' '))

Q041 0 chunk | []
Q043 1 chunk | ['PMC13617261']
   ... ith Monte Carlo dropout‐based uncertainty estimation for CTV segmentation, achieving a DSC of 0.85 ± 0.03 and 95% Hausdorff distance (HD95) of 3.70 ± 0.35 mm. Another study by Zabihollahy et al. evaluated coarse‐to‐fine CNN models for multi‐organ pelvic MRI segmentation, reporting high performance for bladder (DSC 0.93–0.94), rectum (DSC 0.87–0.88), and sigmoid colon (approximately 0.80).
Q048 1 chunk | ['PMC13662491']
   ... nd subsequent neurocognitive, endocrine, and auditory toxicity. Prospective and randomized data have demonstrated that reducing radiation dose to the hippocampi and temporal lobes preserves cognitive function following cranial irradiation. 54 , 55 , 56 There are also well established radiation dose–response relationships for endocrine dysfunction following irradiation of the hypothalamic–pituitary axis 57 


# evaluasi retrieval dan penolakan (butuh variabel dari sel "baca XML, potong menjadi chunk, simpan")

In [10]:
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score

def load_eval(path):
    for enc in ('utf-8-sig', 'cp1252'):          # UTF-8 (dengan/tanpa BOM) dulu, lalu Excel lama
        try:
            df = pd.read_csv(path, encoding=enc); break
        except UnicodeDecodeError:
            continue
    df.columns = [c.strip().lower().lstrip('\ufeff') for c in df.columns]
    return df

ev = load_eval(out / 'eval_questions.csv')
print(list(ev.columns))
ev['gold_pmcid'] = ev['gold_pmcid'].str.strip().replace('PMC13175352', 'PMC13175351')
ev['type'] = ev['type'].replace({'angka': 'numeric', 'metode': 'method', 'perbandingan': 'comparison'})
ev['is_ans'] = ev['answerable'].astype(str).str.lower().isin(['true', '1'])
assert set(ev[ev.is_ans].gold_pmcid) <= {c['pmcid'] for c in chunks}, 'ada gold_pmcid yang tidak ada di korpus'

K = 100
def doc_rank(chunk_ids):
    seen, docs = set(), []
    for j in chunk_ids:
        p = chunks[j]['pmcid']
        if p not in seen: seen.add(p); docs.append(p)
    return docs

def rrf(*rankings, c=60):
    sc = {}
    for r in rankings:
        for pos, j in enumerate(r): sc[j] = sc.get(j, 0) + 1 / (c + pos + 1)
    return [j for j, _ in sorted(sc.items(), key=lambda x: -x[1])]

rows = []
for _, r in ev.iterrows():
    qv = emb_model.encode(['query: ' + r.question], normalize_embeddings=True).astype('float32')
    s, i = index.search(qv, K)
    d_chunks = [int(j) for j in i[0]]
    b_chunks = [int(j) for j in np.argsort(-bm25.get_scores(tok(r.question)))[:K]]
    rows.append(dict(id=r.id, top_dense=float(s[0][0]),
                     dense=doc_rank(d_chunks), bm25=doc_rank(b_chunks), hybrid=doc_rank(rrf(d_chunks, b_chunks))))
res = pd.DataFrame(rows).merge(ev, on='id')

def pos(row, m):
    d = row[m]; return d.index(row.gold_pmcid) + 1 if row.gold_pmcid in d else np.inf

def table(df):
    out_rows = {}
    for m in ['dense', 'bm25', 'hybrid']:
        p = df.apply(lambda r: pos(r, m), axis=1)
        out_rows[m] = {**{f'R@{k}': round((p <= k).mean(), 3) for k in (1, 3, 5, 10)}, 'MRR': round((1 / p).mean(), 3), 'n': len(df)}
    return pd.DataFrame(out_rows).T

a = res[res.is_ans]
print('SEMUA'); print(table(a))
for lang in ['en', 'id']: print('\nBAHASA', lang); print(table(a[a.language == lang]))
for t in sorted(a.type.unique()): print('\nTIPE', t); print(table(a[a.type == t]).loc[['dense', 'hybrid'], ['R@1', 'R@5', 'MRR', 'n']])

print('\nSOAL YANG GAGAL (dense, gold di luar 5 teratas):')
miss = a[a.apply(lambda r: pos(r, 'dense') > 5, axis=1)]
print(miss[['id', 'language', 'type', 'question']].to_string(max_colwidth=90) if len(miss) else '(tidak ada)')

print('\nPENOLAKAN: skor kemiripan tertinggi')
print(res.groupby(['language', 'is_ans']).top_dense.agg(['count', 'mean', 'min', 'max']).round(3))
print('AUROC detektor terjawab:', round(roc_auc_score(res.is_ans, res.top_dense), 3))
ths = np.sort(res.top_dense.unique())
bal = [(t, 0.5 * ((res[res.is_ans].top_dense >= t).mean() + (res[~res.is_ans].top_dense < t).mean())) for t in ths]
t_best, b_best = max(bal, key=lambda x: x[1])
print('ambang terbaik', round(t_best, 3), '| balanced accuracy', round(b_best, 3))

['id', 'question', 'language', 'gold_pmcid', 'reference_answer', 'type', 'answerable']
SEMUA
          R@1   R@3    R@5   R@10    MRR     n
dense   0.850  1.00  1.000  1.000  0.908  40.0
bm25    0.800  0.85  0.975  0.975  0.850  40.0
hybrid  0.875  1.00  1.000  1.000  0.921  40.0

BAHASA en
         R@1  R@3  R@5  R@10    MRR     n
dense   0.95  1.0  1.0   1.0  0.967  20.0
bm25    0.95  1.0  1.0   1.0  0.967  20.0
hybrid  0.90  1.0  1.0   1.0  0.942  20.0

BAHASA id
         R@1  R@3   R@5  R@10    MRR     n
dense   0.75  1.0  1.00  1.00  0.850  20.0
bm25    0.65  0.7  0.95  0.95  0.732  20.0
hybrid  0.85  1.0  1.00  1.00  0.900  20.0

TIPE comparison
        R@1  R@5  MRR    n
dense   1.0  1.0  1.0  2.0
hybrid  1.0  1.0  1.0  2.0

TIPE concept
         R@1  R@5    MRR    n
dense   0.75  1.0  0.833  4.0
hybrid  0.75  1.0  0.875  4.0

TIPE method
          R@1  R@5    MRR     n
dense   1.000  1.0  1.000  13.0
hybrid  0.923  1.0  0.949  13.0

TIPE numeric
          R@1  R@5    MRR     n


# jawaban dengan sitasi, penolakan, dan lembar penilaian manual

In [21]:
ev = load_eval(out / 'eval_questions.csv')
ev['gold_pmcid'] = ev['gold_pmcid'].str.strip().replace('PMC13175352', 'PMC13175351')
ev['type'] = ev['type'].replace({'angka': 'numeric', 'metode': 'method', 'perbandingan': 'comparison'})
ev['is_ans'] = ev['answerable'].astype(str).str.lower().isin(['true', '1'])
bad = ev[ev.question.str.contains(r'\?(?!\s*$)') | ev.reference_answer.fillna('').str.contains(r'\?')]
print(len(bad), 'baris dengan ? mencurigakan'); print(bad[['id']].to_string())

0 baris dengan ? mencurigakan
Empty DataFrame
Columns: [id]
Index: []


In [24]:
import json, re, time
SYS = """You answer questions about medical physics using ONLY the numbered context passages.
Rules:
- If the passages do not contain the answer, set "answerable" to false and "answer" to a short note saying the documents do not cover it. Do not use outside knowledge.
- Otherwise answer concisely (1-3 sentences) and copy numbers exactly as written in the passages.
- Cite the passages you used by number in "citations".
- Answer in the same language as the question.
Reply with JSON only: {"answerable": true|false, "answer": "...", "citations": [1, 2]}"""

def retrieve(q, k=8):
    qv = emb_model.encode(['query: ' + q], normalize_embeddings=True).astype('float32')
    s, i = index.search(qv, 100)
    d = [int(j) for j in i[0]]
    b = [int(j) for j in np.argsort(-bm25.get_scores(tok(q)))[:100]]
    return cap(rrf(d, b), 3)[:k], float(s[0][0])

def answer(q, k=8):
    ids, top = retrieve(q, k)
    ctx = '\n\n'.join(f"[{n+1}] ({chunks[j]['title'][:90]} | {chunks[j]['section'][:50]})\n{chunks[j]['text']}"
                      for n, j in enumerate(ids))
    for t in range(4):
        try:
            r = client.chat.completions.create(model=MODEL, max_tokens=3000, temperature=0, messages=[
                {'role': 'system', 'content': SYS},
                {'role': 'user', 'content': f"Context:\n{ctx}\n\nQuestion: {q}"}])
            d = json.loads(re.search(r'\{.*\}', r.choices[0].message.content, re.S).group(0))
            cited = [ids[n - 1] for n in d.get('citations', []) if isinstance(n, int) and 1 <= n <= len(ids)]
            return dict(pred_answerable=bool(d.get('answerable')), answer=d.get('answer', ''),
                        cited_docs=sorted({chunks[j]['pmcid'] for j in cited}),
                        evidence=' || '.join(chunks[j]['text'][:500] for j in cited),
                        retrieved_docs=doc_rank(ids), top_dense=top)
        except Exception as e:
            print('retry', t, str(e)[:80]); time.sleep(10 * (t + 1))
    return dict(pred_answerable=None, answer='ERROR', cited_docs=[], evidence='', retrieved_docs=[], top_dense=None)

outs = []
for _, r in ev.iterrows():
    outs.append(dict(id=r['id'], **answer(r['question']))); time.sleep(4)
gen = pd.DataFrame(outs).merge(ev, on='id')
gen.to_json(out / 'rag_outputs.jsonl', orient='records', lines=True, force_ascii=False)

err = gen.pred_answerable.isna().sum()
ans, un = gen[gen.is_ans], gen[~gen.is_ans]
print('error:', err)
print('terjawab, model menjawab :', int((ans.pred_answerable == True).sum()), '/', len(ans))
print('tak terjawab, model menolak:', int((un.pred_answerable == False).sum()), '/', len(un))
a2 = ans[ans.pred_answerable == True]
print('sitasi memuat dokumen benar:', int(a2.apply(lambda r: r.gold_pmcid in r.cited_docs, axis=1).sum()), '/', len(a2))
print('\nTAK TERJAWAB yang dijawab model (halusinasi):')
print(un[un.pred_answerable == True][['id', 'question', 'answer']].to_string(max_colwidth=100))
print('\nTERJAWAB yang ditolak model:')
print(ans[ans.pred_answerable == False][['id', 'question']].to_string(max_colwidth=100))

rev = gen[['id', 'language', 'type', 'question', 'reference_answer', 'answer', 'pred_answerable', 'gold_pmcid', 'cited_docs', 'evidence']].copy()
rev['cited_docs'] = rev.cited_docs.apply(lambda x: ';'.join(x))
rev['correct'] = ''; rev['faithful'] = ''; rev['notes'] = ''
rev.to_csv(out / 'manual_review.csv', index=False, encoding='utf-8-sig')

# gerbang topik: soal jelas di luar bidang
off = ["Who won the 2022 World Cup?", "Resep nasi goreng yang enak?", "How do I fix a flat bicycle tire?",
       "Siapa presiden pertama Indonesia?", "What is the capital of Australia?"]
sc = [retrieve(q)[1] for q in off]
print('\nskor di luar topik:', [round(x, 3) for x in sc], '| skor terjawab terendah:', round(res[res.is_ans].top_dense.min(), 3))

error: 0
terjawab, model menjawab : 40 / 40
tak terjawab, model menolak: 10 / 10
sitasi memuat dokumen benar: 40 / 40

TAK TERJAWAB yang dijawab model (halusinasi):
Empty DataFrame
Columns: [id, question, answer]
Index: []

TERJAWAB yang ditolak model:
Empty DataFrame
Columns: [id, question]
Index: []

skor di luar topik: [0.764, 0.744, 0.8, 0.694, 0.753] | skor terjawab terendah: 0.811


# Penilaian

In [9]:
from google.colab import drive
import pandas as pd
from math import sqrt
from pathlib import Path
drive.mount('/content/drive')
out = Path('/content/drive/MyDrive/medphys-rag')

def wilson(k, n, z=1.96):
    p = k / n; d = 1 + z * z / n
    c = (p + z * z / (2 * n)) / d
    h = z * sqrt(p * (1 - p) / n + z * z / (4 * n * n)) / d
    return round(c - h, 3), round(c + h, 3)

try:
    rv = pd.read_csv(out / 'manual_review.csv', encoding='utf-8-sig')
except UnicodeDecodeError:
    rv = pd.read_csv(out / 'manual_review.csv', encoding='cp1252')
rv.columns = [c.strip().lower() for c in rv.columns]

a = rv[rv.gold_pmcid.notna()].copy()                       # baris terjawab = punya gold_pmcid
for c in ['correct', 'faithful']:
    a[c] = a[c].astype(str).str.strip().str.lower()
done = a[a.correct.isin(['yes', 'partial', 'no'])]
n = len(done)
print('dinilai:', n, 'dari', len(a), 'soal terjawab')
assert n > 0, 'kolom correct masih kosong'

for lab, m in [('benar penuh', done.correct == 'yes'), ('benar atau sebagian', done.correct.isin(['yes', 'partial']))]:
    k = int(m.sum()); print(f'{lab}: {k}/{n} = {k/n:.3f} | CI95', wilson(k, n))
f = done[done.faithful.isin(['yes', 'no'])]
k = int((f.faithful == 'yes').sum())
print(f'setia pada sumber: {k}/{len(f)} = {k/len(f):.3f} | CI95', wilson(k, len(f)))
for lang in ['en', 'id']:
    d = done[done.language == lang]
    print(lang, f"benar penuh {int((d.correct == 'yes').sum())}/{len(d)}")
print(done[done.correct != 'yes'][['id', 'question', 'correct', 'notes']].to_string(max_colwidth=80))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
dinilai: 40 dari 40 soal terjawab
benar penuh: 30/40 = 0.750 | CI95 (0.598, 0.858)
benar atau sebagian: 39/40 = 0.975 | CI95 (0.871, 0.996)
setia pada sumber: 38/40 = 0.950 | CI95 (0.835, 0.986)
en benar penuh 12/20
id benar penuh 18/20
      id                                                                         question  correct                                                                            notes
2   Q003  What two patient-specific quality assurance methods were compared to evaluat...  partial  Jawaban menyebut perbandingan EPID dan perangkat detektor konvensional, teta...
10  Q011    What imaging guidance technique was used during Iodine-125 seed implantation?  partial  Jawaban menyebut panduan ultrasound dan CT, tetapi tidak menyertakan penggun...
12  Q013  What statistical method was used to compare dosimetric outcomes between sche...  par